In [ ]:
leftRightString='right'

In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#Importing Libraries
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr

import sys; import os; import time; from datetime import timedelta
import pickle
import h5py
from tqdm import tqdm
import copy
import warnings

In [ ]:
#Importing Additional Libraries
from scipy import stats
from matplotlib.colors import LogNorm

In [ ]:
#MAIN DIRECTORIES
def GetDirectories():
    mainDirectory='/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/'
    mainCodeDirectory=os.path.join(mainDirectory,"Code/CodeFiles/")
    scratchDirectory='/mnt/lustre/koa/scratch/air673/'
    codeDirectory=os.getcwd()
    return mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory

[mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory] = GetDirectories()

In [ ]:
#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from CLASSES_Variable_Calculation import ModelData_Class, SlurmJobArray_Class, DataManager_Class

In [ ]:
#IMPORT FUNCTIONS
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
import FUNCTIONS_Variable_Calculation
from FUNCTIONS_Variable_Calculation import *

In [ ]:
#data loading class
ModelData = ModelData_Class(mainDirectory, scratchDirectory, simulationNumber=4)
#data manager class
DataManager = DataManager_Class(mainDirectory, scratchDirectory, ModelData, dataType="Tracking_Algorithms", dataName="Lagrangian_UpdraftTracking",
                                dtype='float32',codeSection = "Project_Algorithms")

In [ ]:
#data manager class (for saving data)
DataManager_TrackedProfiles = DataManager_Class(mainDirectory, scratchDirectory, ModelData, dataType="Tracked_Profiles", dataName="Tracked_Profiles",
                                dtype='float32',codeSection = "Project_Algorithms")

In [ ]:
#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"3_Project_Algorithms","2_Tracking_Algorithms"))
from CLASSES_TrackingAlgorithms import TrackingAlgorithms_DataLoading_Class, Results_InputOutput_Class, TrackedParcel_Loading_Class

In [ ]:
# IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"3_Project_Algorithms","3_Tracked_Profiles"))
from CLASSES_TrackedProfiles import TrackedProfiles_DataLoading_CLASS,TrackedProfiles_Plotting_CLASS

In [ ]:
#IMPORT FUNCTIONS

import sys
path=os.path.join(mainCodeDirectory,'Functions/')
sys.path.append(path)

import NumericalFunctions
from NumericalFunctions import * # import NumericalFunctions 
import PlottingFunctions
from PlottingFunctions import * # import PlottingFunctions

# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(NumericalFunctions, inspect.isfunction)]
# functions

In [ ]:
#########################################
# DATA LOADING
Apply_FilterOutPriorAscent = False #use for original tracked parcels
# Apply_FilterOutPriorAscent = True #use for tracked parcels removing parcels that come from downdrafts

suffix="_FilterOutPriorAscent" if Apply_FilterOutPriorAscent else ""

In [ ]:
def EntrainmentExtraLoadingSteps(ProfileArraysDictionary_2D,dataName):

    if "Entrainment" in dataName and "Eulerian" not in dataName:
        print('Calculating net entrainment and applying entrainment constant\n')
        #Applying Entrainment Constant
    
        if "DivideMassFlux" not in dataName:
            #data manager class
            DataManager_Entrainment = DataManager_Class(mainDirectory, scratchDirectory, ModelData, dataType="EntrainmentCalculation", dataName="EntrainmentCalculation",dtype='int32')
            
            #getting entrainment constant data
            entrainmentConstant = DataManager_Entrainment.LoadCalculations(
                    DataManager_Entrainment.outputDataDirectory,
                    dataName="EntrainmentConstant",
                    verbose=False,
                )["entrainmentConstant"]
        else:
            entrainmentConstant = 1/np.full(ModelData.Nzh, ModelData.dt)
        
        def ApplyEntrainmentConstant(ProfileArraysDictionary_2D,entrainmentConstant):
            for key1 in ProfileArraysDictionary_2D:  # e.g. 'CL', 'nonCL', etc.
                for key2 in ProfileArraysDictionary_2D[key1]:  # e.g. 'ALL', 'SHALLOW', 'DEEP'
                    for key3 in ProfileArraysDictionary_2D[key1][key2]:  # e.g. 'D_c', 'E_g', etc.
                        for arrayName in ["profile_array","profile_array_left","profile_array_right"]:
                            if arrayName in ProfileArraysDictionary_2D[key1][key2][key3]:
                                arr = ProfileArraysDictionary_2D[key1][key2][key3][arrayName]
                                arr *= entrainmentConstant[np.newaxis,:]
                        # for arrayName in ["profile_array_squares","profile_array_left_squares","profile_array_right_squares"]:
                        #     if arrayName in ProfileArraysDictionary_2D[key1][key2][key3]:
                        #         arr_SE = ProfileArraysDictionary_2D[key1][key2][key3][arrayName]
                        #         arr_SE *= entrainmentConstant[np.newaxis,:]**2
            return ProfileArraysDictionary_2D
    
        
        def CalculateNetEntrainment(ProfileArraysDictionary_2D):
            PROCESSED_string = "PROCESSED_" if "PROCESSED" in dataName else ""
            DivideMassFlux_string = "_DivideMassFlux" if "DivideMassFlux" in dataName else ""
                
            for k1, dict1 in ProfileArraysDictionary_2D.items():
                for k2, data in dict1.items():
                    for suffix in ['g', 'c']:  # Loop through cloudy and general
                        
                        # 1. Subtract the sums (Entrainment - Detrainment)
                        net_sum = (data[f'{PROCESSED_string}E{DivideMassFlux_string}_{suffix}'][f'profile_array_{leftRightString}'] - 
                                   -data[f'{PROCESSED_string}D{DivideMassFlux_string}_{suffix}'][f'profile_array_{leftRightString}'])
                        
                        # 2. Save it back in the exact same format so your existing pipeline can use it
                        data[f'{PROCESSED_string}NET{DivideMassFlux_string}_{suffix}'] = {
                            f'profile_array_{leftRightString}': net_sum,
                            f'profile_array_{leftRightString}_count': data[f'{PROCESSED_string}E{DivideMassFlux_string}_{suffix}']\
                            [f'profile_array_{leftRightString}_count']}
            return ProfileArraysDictionary_2D
    
        def ApplyNegativeToDetrainment(ProfileArraysDictionary_2D):
            PROCESSED_string = "PROCESSED_" if "PROCESSED" in dataName else ""
            DivideMassFlux_string = "_DivideMassFlux" if "DivideMassFlux" in dataName else ""
                
            for k1, dict1 in ProfileArraysDictionary_2D.items():
                for k2, data in dict1.items():
                    for suffix in ['g', 'c']:  # Loop through cloudy and general
                        data[f'{PROCESSED_string}D{DivideMassFlux_string}_{suffix}']\
                        [f'profile_array_{leftRightString}'] *=-1
            return ProfileArraysDictionary_2D
    
        ProfileArraysDictionary_2D = ApplyEntrainmentConstant(ProfileArraysDictionary_2D,entrainmentConstant)
        ProfileArraysDictionary_2D = CalculateNetEntrainment(ProfileArraysDictionary_2D)
        ProfileArraysDictionary_2D = ApplyNegativeToDetrainment(ProfileArraysDictionary_2D)
    return ProfileArraysDictionary_2D

In [ ]:
dataName="Variables"
ProfileArraysDictionary_2D_Variables = TrackedProfiles_DataLoading_CLASS.LoadProfile(ModelData,DataManager_TrackedProfiles, dataName, t=f'combined_TZContour{suffix}')

In [ ]:
dataName="PROCESSED_Entrainment_DivideMassFlux"
ProfileArraysDictionary_2D_Entrainment = TrackedProfiles_DataLoading_CLASS.LoadProfile(ModelData,DataManager_TrackedProfiles, dataName, t=f'combined_TZContour{suffix}')
ProfileArraysDictionary_2D_Entrainment=EntrainmentExtraLoadingSteps(ProfileArraysDictionary_2D_Entrainment,dataName)

In [ ]:
dataName="UpdraftArea"
ProfileArraysDictionary_2D_UpdraftArea = TrackedProfiles_DataLoading_CLASS.LoadProfile(ModelData,DataManager_TrackedProfiles, dataName, t=f'combined_TZContour{suffix}')

In [ ]:
_,LevelsDictionary = TrackedParcel_Loading_Class.LoadingSubsetParcelData(ModelData,DataManager,
                                                         Results_InputOutput_Class)

In [ ]:
#########################################
#CALCULATION FUNCTIONS

In [ ]:
def GetArrays(ProfileArraysDictionary_2D,parcelType,parcelDepth,varName):
    profile = ProfileArraysDictionary_2D[parcelType][parcelDepth][varName][f'profile_array_{leftRightString}'].copy()
    count = ProfileArraysDictionary_2D[parcelType][parcelDepth][varName][f'profile_array_{leftRightString}_count'].copy()
    return profile,count

def ApplyCountThreshold(profile1, count1, profile2, count2, countThreshold=0):
    if countThreshold==0:
        return profile1, count1, profile2, count2
        
    # Create separate masks for each dataset
    mask1 = count1 <= countThreshold
    mask2 = count2 <= countThreshold
    
    outputs = []
    for i, arr in enumerate([profile1, count1, profile2, count2]):
        arr_copy = arr.copy().astype(float)
        
        # If it's the first two (profile1/count1), use mask1
        # Otherwise (profile2/count2), use mask2
        current_mask = mask1 if i < 2 else mask2
        
        arr_copy[current_mask] = np.nan
        outputs.append(arr_copy)
    return outputs # profile1, count1, profile2, count2
# def ApplyCountThreshold(profile1,count1,profile2,count2,countThreshold=10): #disabled for now
#     if countThreshold==0:      
#         return profile1, count1, profile2, count2
#     mask = (count1 <= countThreshold) | (count2 <= countThreshold)
#     outputs = []
#     for arr in [profile1, count1, profile2, count2]:
#         arr_copy = arr.copy().astype(float) # Ensure float for NaN support
#         arr_copy[mask] = np.nan
#         outputs.append(arr_copy)
#     return outputs #profile1,count1,profile2,count2

def NanDivide(a, b):
    return np.divide(a, b, where=b!=0, out=np.full_like(a, np.nan, dtype=float))
# def NanSubtract(a, b):
#     return np.subtract(a, b)
def NanSubtract(a, b): #*subtracting the magnitude instead
    return np.subtract(np.abs(a), np.abs(b))

def TakeMean(profile,count, 
             varName,meanDataDictionary=None):
    mean_profile = NanDivide(profile, count)

    #perturbations calculated here
    if meanDataDictionary is not None:
        mean_profile -= meanDataDictionary[varKeyMatchDictionary[varName]]
    return mean_profile

def GetTimeIndices(t1_hrs,t2_hrs):
    time_hrs = np.arange(ModelData.Ntime)*ModelData.dt/3600+6
    t1 = np.abs(time_hrs - t1_hrs).argmin()
    t2 = np.abs(time_hrs - t2_hrs).argmin()
    return t1,t2

In [ ]:
#########################################
#PLOTTING CONFIG

In [ ]:
#variable plotting dictionary

#Variables
variableInfo = {
    f"QV": {
        "label": r"$q_v$",
        "units": r"($g\ kg^{-1}$)",
        "multiplier": 1e3
    }, 
    f"QCQI": {
        "label": r"$q_c+q_i$",
        "units": r"($g\ kg^{-1}$)",
        "multiplier": 1e3
    }, 
    f"RH_vapor": {
        "label": r"$RH_v$",
        "units": "(%)",
        "multiplier": 1e2
    }, 
    f"W": {
        "label": "w",
        "units": r"($m\ s^{-1}$)",
        "multiplier": 1
    },
    f"VMF_g": {
        "label": r"$VMF_g$",
        "units": r"($kg\ m^{-2}\ s^{-1}$)",
        "multiplier": 1
    },
    f"VMF_c": {
        "label": r"$VMF_c$",
        "units": r"($kg\ m^{-2}\ s^{-1}$)",
        "multiplier": 1
    },
    f"HMC": {
        "label": "HMC",
        "units": r"($g\ kg^{-1}\ s^{-1}$)",
        "multiplier": 1e3
    },    
    f"THETA": {
        "label": r"$\theta$",
        "units": "(K)",
        "multiplier": 1
    },
    f"THETA_v": {
        "label": r"$\theta_v$",
        "units": "(K)",
        "multiplier": 1
    },
    f"THETA_e": {
        "label": r"$\theta_e$",
        "units": "(K)",
        "multiplier": 1
    },
    f"BUOYANCY": {
        "label": r"$B$",
        "units": r"($m\ s^{-1}/s$)",
        "multiplier": 1
    },
    f"MSE": {
        "label": r"$MSE$",
        "units": "(K)",
        "multiplier": 1/1005.7
    }
}

#UpdraftArea
variableInfo.update({f"UpdraftArea_g": {"label": r"$UpdraftArea_g$",
                                        "units": r"($km^2$)",
                                        "multiplier": 1 / (ModelData.dx**2)},
                     f"UpdraftArea_c": {"label": r"$UpdraftArea_c$",
                                        "units": r"($km^2$)",
                                        "multiplier": 1 / (ModelData.dx**2)}
                    })

#Budgets
variableInfo.update({f"WB_BUOY": {"label": r"$WB_BUOY$",
                                        "units": r"($m\,s^{-1}/s$)",
                                        "multiplier": 1},
                     f"WB_PGRAD": {"label": r"$WB_PGRAD$",
                                        "units": r"($m\,s^{-1}/s$)",
                                        "multiplier": 1},
                     f"QVB_VADV": {"label": r"$QVB_VADV$",
                                        "units": r"($g\,kg^{-1}/s$)",
                                        "multiplier": 1e3},
                     f"QVB_MP": {"label": r"$QVB_MP$",
                                        "units": r"($g\,kg^{-1}/s$)",
                                        "multiplier": 1e3},
                     f"PTB_MP": {"label": r"$PTB_MP$",
                                        "units": r"($K/s$)",
                                        "multiplier": 1},
                     f"PTB_VADV": {"label": r"$PTB_VADV$",
                                        "units": r"($K/s$)",
                                        "multiplier": 1},
                    })

#Lagrangian Entrainment
variableInfo.update({f"NET_g": {"label": r"$NET_g$",
                                          "units": r"($kg\ m^{-3}\ s^{-1}$)",
                                          "multiplier": 1},
                     f"NET_c": {"label": r"$NET_c$",
                                          "units": r"($kg\ m^{-3}\ s^{-1}$)",
                                          "multiplier": 1},
                     f"PROCESSED_NET_g": {"label": r"$PROCESSED\_NET\_g$",
                                          "units": r"($kg\ m^{-3}\ s^{-1}$)",
                                          "multiplier": 1},
                     f"PROCESSED_NET_c": {"label": r"$PROCESSED\_NET_c$",
                                          "units": r"($kg\ m^{-3}\ s^{-1}$)",
                                          "multiplier": 1},
                     f"PROCESSED_NET_DivideMassFlux_g": {"label": r"$PROCESSED\_NET_g$",
                                                         "units": r"($km^{-1}$)",
                                                         "multiplier": 1},
                     f"PROCESSED_NET_DivideMassFlux_c": {"label": r"$PROCESSED\_NET_c$",
                                                         "units": r"($km^{-1}$)",
                                                         "multiplier": 1},
                     f"E_g": {"label": r"$NET_g$",
                                          "units": r"($kg\ m^{-3}\ s^{-1}$)",
                                          "multiplier": 1},
                     f"E_c": {"label": r"$NET_c$",
                                          "units": r"($kg\ m^{-3}\ s^{-1}$)",
                                          "multiplier": 1},
                     f"PROCESSED_E_g": {"label": r"$PROCESSED\_D\_g$",
                                          "units": r"($kg\ m^{-3}\ s^{-1}$)",
                                          "multiplier": 1},
                     f"PROCESSED_E_c": {"label": r"$PROCESSED\_D_c$",
                                          "units": r"($kg\ m^{-3}\ s^{-1}$)",
                                          "multiplier": 1},
                     f"PROCESSED_E_DivideMassFlux_g": {"label": r"$PROCESSED\_D_g$",
                                                         "units": r"($km^{-1}$)",
                                                         "multiplier": 1},
                     f"PROCESSED_E_DivideMassFlux_c": {"label": r"$PROCESSED\_D_c$",
                                                         "units": r"($km^{-1}$)",
                                                         "multiplier": 1},
                      f"D_g": {"label": r"$D_g$",
                                          "units": r"($kg\ m^{-3}\ s^{-1}$)",
                                          "multiplier": 1},
                     f"D_c": {"label": r"$D_c$",
                                          "units": r"($kg\ m^{-3}\ s^{-1}$)",
                                          "multiplier": 1},
                     f"PROCESSED_D_g": {"label": r"$PROCESSED\_D\_g$",
                                          "units": r"($kg\ m^{-3}\ s^{-1}$)",
                                          "multiplier": 1},
                     f"PROCESSED_D_c": {"label": r"$PROCESSED\_D_c$",
                                          "units": r"($kg\ m^{-3}\ s^{-1}$)",
                                          "multiplier": 1},
                     f"PROCESSED_D_DivideMassFlux_g": {"label": r"$PROCESSED\_D_g$",
                                                         "units": r"($km^{-1}$)",
                                                         "multiplier": 1},
                     f"PROCESSED_D_DivideMassFlux_c": {"label": r"$PROCESSED\_D_c$",
                                                         "units": r"($km^{-1}$)",
                                                         "multiplier": 1}
                    })

#Lagrangian Entrainment
variableInfo.update({f"NET_g_eulerian": {"label": r"$NET_g$",
                                          "units": r"($kg\ m^{-3}\ s^{-1}$)",
                                          "multiplier": 1},
                     f"NET_c_eulerian": {"label": r"$NET_c$",
                                          "units": r"($kg\ m^{-3}\ s^{-1}$)",
                                          "multiplier": 1},
                     f"NET_g_eulerian_DivideMassFlux": {"label": r"$PROCESSED\_NET_g$",
                                                         "units": r"($km^{-1}$)",
                                                         "multiplier": 1},
                     f"NET_c_eulerian_DivideMassFlux": {"label": r"$PROCESSED\_NET_c$",
                                                         "units": r"($km^{-1}$)",
                                                         "multiplier": 1}
                    })

In [ ]:
#########################################
#PLOTTING FUNCTIONS

In [ ]:
#MakeContour
time=np.arange(ModelData.Ntime)*ModelData.dt/3600+6
def MakeContour(varName,plot_data,method='minmax',xData=time,yData=ModelData.zh,
                cmap='viridis',fig=None,axis=None):
    if axis is None:
        fig,axis=plt.subplots()
    [levels,extend]=GetLevels(dataName,varName,plot_data,method=method)
    cf=axis.contourf(xData,yData,plot_data.T,levels=levels,extend=extend,cmap=cmap)
    cbar=fig.colorbar(cf,ax=axis,
                      label=variableInfo[varName]['label']\
                      +" "+variableInfo[varName]['units'])
    axis.set_ylim(top=10)
    return cbar
def GetLevels(dataName,varName,plot_data,method='minmax',nLevels=41):
    if method=='percentile':
        vmax = np.nanpercentile(plot_data,95)
        vmin = -vmax
    elif method=='minmax':
        vmin=np.nanmin(plot_data)
        vmax=np.nanmax(plot_data)
    if varName == "QCQI":
        vmin = 1e-6
    elif "UpdraftArea" in varName:
        vmin=0
    levels=np.linspace(vmin,vmax,nLevels)

    extend='max' if "UpdraftArea" in varName else 'both'
    return levels,extend

def MakeScatter(varName_1,varName_2,mean_profile_1,mean_profile_2,
                fig=None,axis=None,color='black',extent_label=None):
    a = mean_profile_1.flatten()
    b = mean_profile_2.flatten()
    
    # Remove NaN/inf values
    mask = np.isfinite(a) & np.isfinite(b)
    if np.any(mask):
        a_clean, b_clean = a[mask], b[mask]
        
        if axis is None:
            axis=plt.gca()
        # Linear regression with stats
        slope, intercept, r_value, p_value, std_err = stats.linregress(a_clean, b_clean)
        
        axis.scatter(a_clean, b_clean, s=0.5)
        if extent_label is None:
            label=f'y = {slope:.1e}x + {intercept:.1e}\nR²={r_value**2:.3f}, p={p_value:.2e}'
        else:
            label=f'{extent_label}: y = {slope:.1e}x + {intercept:.1e}\nR²={r_value**2:.3f}, p={p_value:.2e}'
        axis.plot(a_clean, slope * a_clean + intercept, color=color,
                 label=label)
        axis.axhline(0,color='gray',zorder=10)
        axis.legend()
        
        axis.set_xlabel(variableInfo[varName_1]['label']\
                        +" "+variableInfo[varName_1]['units'])
        axis.set_ylabel(variableInfo[varName_2]['label']\
                        +" "+variableInfo[varName_2]['units'])
        apply_scientific_notation([axis],dimension='y')

In [ ]:
def GetCloudBase(ProfileArraysDictionary_2D_Variables,
                 parcelType="CL",parcelDepth="SHALLOW"):
    varName="QCQI"
    data=GetArrays(ProfileArraysDictionary_2D_Variables,parcelType,parcelDepth,varName)
    profile=data[0];count=data[1]
    mean_profile = TakeMean(profile,count,varName)*variableInfo[varName]['multiplier']

    time_indices, z_indices = np.where(mean_profile >= 1e-6)
    
    cloud_base_zInd = np.full(mean_profile.shape[0], np.nan)
    
    # Vectorized: get minimum z_index per time using np.minimum.at
    min_z = np.full(mean_profile.shape[0], mean_profile.shape[1])  # initialize with max z index
    np.minimum.at(min_z, time_indices, z_indices)
    
    # Only fill times that had at least one valid z (NaN where whole column is nan)
    has_cloud = np.isin(np.arange(mean_profile.shape[0]), time_indices)
    cloud_base_zInd[has_cloud] = min_z[has_cloud]
    cloud_base_zInd[has_cloud] = min_z[has_cloud] .astype(int)
    return cloud_base_zInd

def ApplyCloudBaseToArray(mean_profile, cloud_base_zInd, mask="below"):
    if mask == "below":
        # If no cloud base exists, 
        #mask the ENTIRE column by setting the base above the max z-index
        fill = mean_profile.shape[1] + 1                      
    elif mask == "above":
        # If no cloud base exists, 
        # mask the ENTIRE column by setting the base below the min z-index
        fill = -1 
    
    valid_base = np.where(np.isnan(cloud_base_zInd), fill, cloud_base_zInd).astype(int)
    z_grid = np.arange(mean_profile.shape[1])[np.newaxis, :]
    min_z_grid = valid_base[:, np.newaxis]

    if mask == "below":
        mean_profile[z_grid < min_z_grid] = np.nan
    elif mask == "above":
        mean_profile[z_grid >= min_z_grid] = np.nan

    return mean_profile

def PlotCloudBase(cloud_base_zInd,fig=None,axis=None,color='black'):
    if axis is None:
        fig,axis=plt.subplots()
    
    valid = ~np.isnan(cloud_base_zInd)
    cloud_base_height = np.full(len(cloud_base_zInd), np.nan)
    cloud_base_height[valid] = ModelData.zh[cloud_base_zInd[valid].astype(int)]
    
    axis.plot(time, cloud_base_height, color=color, linewidth=1.5, zorder=10)

def GetLFC(LevelsDictionary):
    LFC_zInd = [np.nan if np.isnan(LFC) else abs(ModelData.zh - LFC).argmin() for LFC in LevelsDictionary["LFC_profile"]]
    return np.array(LFC_zInd)

In [ ]:
#########################################
#Comparing Updraft Area and Fractional Entrainment 

In [ ]:
def MakePlot(varName_1="UpdraftArea_c",varName_2="PROCESSED_NET_DivideMassFlux_c",
             parcelType="CL",parcelDepth="SHALLOW"):

    cloud_base_zInd=GetCloudBase(ProfileArraysDictionary_2D_Variables,
                 parcelType=parcelType,parcelDepth=parcelDepth)
    cloud_base_mask = "below" if "_c" in varName_1 else "above"
    
    #First Variable
    data=GetArrays(ProfileArraysDictionary_2D_UpdraftArea,parcelType,parcelDepth,varName_1)
    profile=data[0];count=data[1]
    mean_profile_1 = TakeMean(profile,count,varName_1)*variableInfo[varName_1]['multiplier']
    mean_profile_1 = ApplyCloudBaseToArray(mean_profile_1,cloud_base_zInd,mask=cloud_base_mask)
    
    #Second Variable
    data=GetArrays(ProfileArraysDictionary_2D_Entrainment,parcelType,parcelDepth,varName_2)\
    *variableInfo[varName_2]['multiplier']
    profile=data[0];count=data[1]
    mean_profile_2 = TakeMean(profile,count,varName_2)
    mean_profile_2 = ApplyCloudBaseToArray(mean_profile_2,cloud_base_zInd,mask=cloud_base_mask)
    

    #Plotting Each
    fig,axis = plt.subplots(1,3,figsize=(18,4))
    cbar1=MakeContour(varName_1,plot_data=mean_profile_1,
                      fig=fig,axis=axis[0],method='percentile')
    cbar2=MakeContour(varName_2,plot_data=mean_profile_2,cmap='RdBu_r',
                      fig=fig,axis=axis[1],method='percentile')
    apply_scientific_notation_colorbar([cbar2],decimals=3)

    #Plotting Comparison
    if "_g" in varName_1:
        MakeScatter(varName_1,varName_2,mean_profile_1,mean_profile_2,fig=fig,axis=axis[2])
    elif "_c" in varName_1:
        zInd_5km = np.abs(ModelData.zh-5).argmin()
        
        mean_profile_1_scatter = mean_profile_1[:,:zInd_5km+1].copy()
        mean_profile_2_scatter = mean_profile_2[:,:zInd_5km+1].copy()
        MakeScatter(varName_1,varName_2,mean_profile_1_scatter,mean_profile_2_scatter,
                      fig=fig,axis=axis[2],color='black',extent_label='below 5 km')
        mean_profile_1_scatter = mean_profile_1[:,zInd_5km:].copy()
        mean_profile_2_scatter = mean_profile_2[:,zInd_5km:].copy()
        MakeScatter(varName_1,varName_2,mean_profile_1_scatter,mean_profile_2_scatter,
                      fig=fig,axis=axis[2],color='red',extent_label='above 5 km')

def MakeAllPlots(parcelType="CL",entrainmentType="NET"):
    
    #SHALLOW
    parcelDepth="SHALLOW"
    #general updraft
    varName_1="UpdraftArea_g";varName_2=f"PROCESSED_{entrainmentType}_DivideMassFlux_g"
    MakePlot(varName_1=varName_1,varName_2=varName_2,
             parcelType=parcelType,parcelDepth=parcelDepth)
    # #cloudy updraft
    # varName_1="UpdraftArea_c";varName_2=f"PROCESSED_{entrainmentType}_DivideMassFlux_c"
    # MakePlot(varName_1=varName_1,varName_2=varName_2,
    #          parcelType=parcelType,parcelDepth=parcelDepth)
    
    #DEEP
    parcelDepth="DEEP"
    #general updraft
    varName_1="UpdraftArea_g";varName_2=f"PROCESSED_{entrainmentType}_DivideMassFlux_g"
    MakePlot(varName_1=varName_1,varName_2=varName_2,
             parcelType=parcelType,parcelDepth=parcelDepth)
    # #cloudy updraft
    # varName_1="UpdraftArea_c";varName_2=f"PROCESSED_{entrainmentType}_DivideMassFlux_c"
    # MakePlot(varName_1=varName_1,varName_2=varName_2,
    #          parcelType=parcelType,parcelDepth=parcelDepth)

In [ ]:
MakeAllPlots(parcelType="CL",entrainmentType="NET")
MakeAllPlots(parcelType="CL",entrainmentType="E")
# MakeAllPlots(parcelType="CL",entrainmentType="D")

In [ ]:
MakeAllPlots(parcelType="nonCL",entrainmentType="NET")
MakeAllPlots(parcelType="nonCL",entrainmentType="E")
# MakeAllPlots(parcelType="nonCL",entrainmentType="D")

In [ ]:
#########################################
#Variables at Cloud Base

In [ ]:
def MakeDifferencePlot(varName,parcelTypes=["CL","nonCL"],
                       fig=None,axis=None):
    
    #Getting Variable
    data=GetArrays(ProfileArraysDictionary_2D_Variables,parcelTypes[0],parcelDepth,varName)
    profile=data[0];count=data[1]
    mean_profile1 = TakeMean(profile,count,varName)*variableInfo[varName]['multiplier']
    cloud_base_zInd1=GetCloudBase(ProfileArraysDictionary_2D_Variables,
                             parcelType=parcelTypes[0],parcelDepth=parcelDepth)
    
    parcelType="nonCL"
    #Getting Variable
    data=GetArrays(ProfileArraysDictionary_2D_Variables,parcelTypes[1],parcelDepth,varName)
    profile=data[0];count=data[1]
    mean_profile2 = TakeMean(profile,count,varName)*variableInfo[varName]['multiplier']
    cloud_base_zInd2=GetCloudBase(ProfileArraysDictionary_2D_Variables,
                             parcelType=parcelTypes[1],parcelDepth=parcelDepth)

    if axis is None:
        fig,axis=plt.subplots()
    
    MakeContour(varName,mean_profile1-mean_profile2,xData=time,yData=ModelData.zh,
            cmap='RdBu_r',fig=fig,axis=axis,method='percentile')
    PlotCloudBase(cloud_base_zInd1,fig=fig,axis=axis,color='black')
    PlotCloudBase(cloud_base_zInd2,fig=fig,axis=axis,color='blue')


def MakePlot(parcelType="CL",parcelDepth="SHALLOW",varName='W',
             fig=None,axis=None,color='black',label=""):

    if axis is None:
        fig,axis=plt.subplots()
    cloud_base_zInd=GetCloudBase(ProfileArraysDictionary_2D_Variables,
                                 parcelType=parcelType,parcelDepth=parcelDepth)
    
    #Getting Variable
    data=GetArrays(ProfileArraysDictionary_2D_Variables,parcelType,parcelDepth,varName)
    profile=data[0];count=data[1]
    mean_profile = TakeMean(profile,count,varName)*variableInfo[varName]['multiplier']
    
    #Getting Data at Cloudbase 
    valid = ~np.isnan(cloud_base_zInd)
    result = np.full(mean_profile.shape[0], np.nan)
    result[valid] = mean_profile[valid, cloud_base_zInd[valid].astype(int)]
    axis.plot(time,result,color=color,label=label)
    axis.set_xlabel('time (hrs)')
    axis.legend()
    axis.set_ylabel(variableInfo[varName]['label']\
                +" "+variableInfo[varName]['units'])
    apply_scientific_notation([axis],dimension='y')

def MakeAllPlots(parcelTypes,parcelDepth,varName):
    fig,axes=plt.subplots(1,2,figsize=(14,4))
    MakeDifferencePlot(varName,parcelTypes=parcelTypes,fig=fig,axis=axes[0])
    MakePlot(parcelType=parcelTypes[0],parcelDepth=parcelDepth,varName=varName,
             fig=fig,axis=axes[1],color='black',label=parcelTypes[0])
    MakePlot(parcelType=parcelTypes[1],parcelDepth=parcelDepth,varName=varName,
             fig=fig,axis=axes[1],color='blue',label=parcelTypes[1])

In [ ]:
parcelTypes=["CL","nonCL"]; parcelDepth="SHALLOW"
varName='THETA_v'
MakeAllPlots(parcelTypes,parcelDepth,varName)
varName='QV'
MakeAllPlots(parcelTypes,parcelDepth,varName)
varName='W'
MakeAllPlots(parcelTypes,parcelDepth,varName)

In [ ]:
#########################################
#Variables at LFC

In [ ]:
def MakeDifferencePlot(varName,parcelTypes=["CL","nonCL"],
                       fig=None,axis=None):
    
    #Getting Variable
    data=GetArrays(ProfileArraysDictionary_2D_Variables,parcelTypes[0],parcelDepth,varName)
    profile=data[0];count=data[1]
    mean_profile1 = TakeMean(profile,count,varName)*variableInfo[varName]['multiplier']
    LFC_zInd1=GetLFC(LevelsDictionary)
    
    parcelType="nonCL"
    #Getting Variable
    data=GetArrays(ProfileArraysDictionary_2D_Variables,parcelTypes[1],parcelDepth,varName)
    profile=data[0];count=data[1]
    mean_profile2 = TakeMean(profile,count,varName)*variableInfo[varName]['multiplier']
    LFC_zInd2=GetLFC(LevelsDictionary)

    if axis is None:
        fig,axis=plt.subplots()
    
    MakeContour(varName,mean_profile1-mean_profile2,xData=time,yData=ModelData.zh,
            cmap='RdBu_r',fig=fig,axis=axis,method='percentile')
    PlotCloudBase(LFC_zInd1,fig=fig,axis=axis,color='black')
    PlotCloudBase(LFC_zInd2,fig=fig,axis=axis,color='blue')


def MakePlot(parcelType="CL",parcelDepth="SHALLOW",varName='W',
             fig=None,axis=None,color='black',label=""):

    if axis is None:
        fig,axis=plt.subplots()
    LFC_zInd=GetLFC(LevelsDictionary)
    
    #Getting Variable
    data=GetArrays(ProfileArraysDictionary_2D_Variables,parcelType,parcelDepth,varName)
    profile=data[0];count=data[1]
    mean_profile = TakeMean(profile,count,varName)*variableInfo[varName]['multiplier']
    
    #Getting Data at Cloudbase 
    valid = ~np.isnan(LFC_zInd)
    result = np.full(mean_profile.shape[0], np.nan)
    result[valid] = mean_profile[valid, LFC_zInd[valid].astype(int)]
    axis.plot(time,result,color=color,label=label)
    axis.set_xlabel('time (hrs)')
    axis.legend()
    axis.set_ylabel(variableInfo[varName]['label']\
                +" "+variableInfo[varName]['units'])
    apply_scientific_notation([axis],dimension='y')

def MakeAllPlots(parcelTypes,parcelDepth,varName):
    fig,axes=plt.subplots(1,2,figsize=(14,4))
    MakeDifferencePlot(varName,parcelTypes=parcelTypes,fig=fig,axis=axes[0])
    MakePlot(parcelType=parcelTypes[0],parcelDepth=parcelDepth,varName=varName,
             fig=fig,axis=axes[1],color='black',label=parcelTypes[0])
    MakePlot(parcelType=parcelTypes[1],parcelDepth=parcelDepth,varName=varName,
             fig=fig,axis=axes[1],color='blue',label=parcelTypes[1])

In [ ]:
parcelTypes=["CL","nonCL"]; parcelDepth="SHALLOW"
varName='THETA_v'
MakeAllPlots(parcelTypes,parcelDepth,varName)
varName='QV'
MakeAllPlots(parcelTypes,parcelDepth,varName)
varName='W'
MakeAllPlots(parcelTypes,parcelDepth,varName)